# Bonus Unit 2. Agent Observability and Evaluation

In [1]:
# Mantén False para el recorrido que funciona sin dependencias externas.
run_external = False

def install_packages(*args):
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)


## Monitoring and evaluating agents

### Práctica local

In [2]:
from time import perf_counter

traces = []

def traced_call(name, function, *args):
    start = perf_counter()
    result = function(*args)
    traces.append({
        "name": name,
        "input": args,
        "output": result,
        "duration_ms": (perf_counter() - start) * 1000,
    })
    return result

def add(a, b):
    return a + b

assert traced_call("add", add, 20, 22) == 42
assert traced_call("add", add, 4, 7) == 11
assert all(trace["duration_ms"] >= 0 for trace in traces)
for trace in traces:
    print(f"{trace['name']}{trace['input']} = {trace['output']}")


add(20, 22) = 42
add(4, 7) = 11


In [3]:
expected_outputs = [42, 11]
assert len(traces) == len(expected_outputs)
matches = [trace["output"] == expected for trace, expected in zip(traces, expected_outputs)]
assert all(matches)
print("Trazas registradas:", len(traces))
print(f"Coincidencia sobre ejemplos locales: {sum(matches)}/{len(matches)}")


Trazas registradas: 2
Coincidencia sobre ejemplos locales: 2/2


### Ejemplo con servicios externos

In [4]:
if run_external:
    install_packages(*['langfuse', 'smolagents[telemetry]', 'openinference-instrumentation-smolagents', 'datasets', 'smolagents[gradio]', 'gradio', '--upgrade'])


In [5]:
if run_external:
    import os
    from google.colab import userdata

    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")


In [6]:
if run_external:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [7]:
if run_external:
    from langfuse import get_client

    langfuse = get_client()

    # Verify connection
    if langfuse.auth_check():
        print("Langfuse client is authenticated and ready!")
    else:
        print("Authentication failed. Please check your credentials and host.")


In [8]:
if run_external:
    from openinference.instrumentation.smolagents import SmolagentsInstrumentor

    SmolagentsInstrumentor().instrument()


In [9]:
if run_external:
    from smolagents import InferenceClientModel, CodeAgent

    # Create a simple agent to test instrumentation
    agent = CodeAgent(
        tools=[],
        model=InferenceClientModel()
    )

    agent.run("1+1=")


In [10]:
if run_external:
    from smolagents import (CodeAgent, DuckDuckGoSearchTool, InferenceClientModel)

    search_tool = DuckDuckGoSearchTool()
    agent = CodeAgent(tools=[search_tool], model=InferenceClientModel())

    agent.run("How many Rubik's Cubes could you fit inside the Notre Dame Cathedral?")


In [11]:
if run_external:
    from smolagents import (CodeAgent, DuckDuckGoSearchTool, InferenceClientModel)

    search_tool = DuckDuckGoSearchTool()
    agent = CodeAgent(
        tools=[search_tool],
        model=InferenceClientModel()
    )

    with langfuse.start_as_current_span(
        name="Smolagent-Trace",
        ) as span:

        # Run your application here
        response = agent.run("What is the capital of Germany?")

        # Pass additional attributes to the span
        span.update_trace(
            input="What is the capital of Germany?",
            output=response,
            user_id="smolagent-user-123",
            session_id="smolagent-session-123456789",
            tags=["city-question", "testing-agents"],
            metadata={"email": "user@langfuse.com"},
            )

    # Flush events in short-lived applications
    langfuse.flush()


In [12]:
if run_external:
    import gradio as gr
    from smolagents import (CodeAgent, InferenceClientModel)
    from langfuse import get_client

    langfuse = get_client()

    model = InferenceClientModel()
    agent = CodeAgent(tools=[], model=model, add_base_tools=True)

    trace_id = None

    def respond(prompt, history):
        with langfuse.start_as_current_span(
            name="Smolagent-Trace"):

            # Run your application here
            output = agent.run(prompt)

            global trace_id
            trace_id = langfuse.get_current_trace_id()

        history.append({"role": "assistant", "content": str(output)})
        return history

    def handle_like(data: gr.LikeData):
        # For demonstration, we map user feedback to a 1 (like) or 0 (dislike)
        if data.liked:
            langfuse.create_score(
                value=1,
                name="user-feedback",
                trace_id=trace_id
            )
        else:
            langfuse.create_score(
                value=0,
                name="user-feedback",
                trace_id=trace_id
            )

    with gr.Blocks() as demo:
        chatbot = gr.Chatbot(label="Chat", type="messages")
        prompt_box = gr.Textbox(placeholder="Type your message...", label="Your message")

        # When the user presses 'Enter' on the prompt, we run 'respond'
        prompt_box.submit(
            fn=respond,
            inputs=[prompt_box, chatbot],
            outputs=chatbot
        )

        # When the user clicks a 'like' button on a message, we run 'handle_like'
        chatbot.like(handle_like, None, None)

    demo.launch()


In [13]:
if run_external:
    # Example: Checking if the agent’s output is toxic or not.
    from smolagents import (CodeAgent, DuckDuckGoSearchTool, InferenceClientModel)

    search_tool = DuckDuckGoSearchTool()
    agent = CodeAgent(tools=[search_tool], model=InferenceClientModel())

    agent.run("Can eating carrots improve your vision?")


In [14]:
if run_external:
    import pandas as pd
    from datasets import load_dataset

    # Fetch GSM8K from Hugging Face
    dataset = load_dataset("openai/gsm8k", 'main', split='train')
    df = pd.DataFrame(dataset)
    print("First few rows of GSM8K dataset:")
    print(df.head())


In [15]:
if run_external:
    from langfuse import get_client
    langfuse = get_client()

    langfuse_dataset_name = "gsm8k_dataset_huggingface"

    # Create a dataset in Langfuse
    langfuse.create_dataset(
        name=langfuse_dataset_name,
        description="GSM8K benchmark dataset uploaded from Huggingface",
        metadata={
            "date": "2025-03-10",
            "type": "benchmark"
        }
    )


In [16]:
if run_external:
    for idx, row in df.iterrows():
        langfuse.create_dataset_item(
            dataset_name=langfuse_dataset_name,
            input={"text": row["question"]},
            expected_output={"text": row["answer"]},
            metadata={"source_index": idx}
        )
        if idx >= 9: # Upload only the first 10 items for demonstration
            break


In [17]:
if run_external:
    from opentelemetry.trace import format_trace_id
    from smolagents import (CodeAgent, InferenceClientModel, LiteLLMModel)
    from langfuse import get_client

    langfuse = get_client()


    # Example: using InferenceClientModel or LiteLLMModel to access openai, anthropic, gemini, etc. models:
    model = InferenceClientModel()

    agent = CodeAgent(
        tools=[],
        model=model,
        add_base_tools=True
    )

    dataset_name = "gsm8k_dataset_huggingface"
    current_run_name = "smolagent-notebook-run-01" # Identifies this specific evaluation run

    # Assume 'run_smolagent' is your instrumented application function
    def run_smolagent(question):
        with langfuse.start_as_current_generation(name="qna-llm-call") as generation:
            # Simulate LLM call
            result = agent.run(question)

            # Update the trace with the input and output
            generation.update_trace(
                input= question,
                output=result,
            )

            return result

    dataset = langfuse.get_dataset(name=dataset_name) # Fetch your pre-populated dataset

    for item in dataset.items:

        # Use the item.run() context manager
        with item.run(
            run_name=current_run_name,
            run_metadata={"model_provider": "Hugging Face", "temperature_setting": 0.7},
            run_description="Evaluation run for GSM8K dataset"
        ) as root_span: # root_span is the root span of the new trace for this item and run.
            # All subsequent langfuse operations within this block are part of this trace.

            # Call your application logic
            generated_answer = run_smolagent(question=item.input["text"])

            print(item.input)
